# Historical archive: preprocessing ecthr_a

**Status: not independently reproduced.**

Checkpoint / purpose: `Analysis / preprocessing`.

Historical code, not independently reproduced. Saved settings can differ from report labels; notebook filenames are not authoritative configurations. Article descriptions map IDs to Articles 1–10 incorrectly. Historic preprocessing is retained for provenance; use the new preparation pipeline for corrected labels.

Publication edits only: credentials use `HF_TOKEN`; personal absolute paths were relocated; saved outputs, attachments and runtime metadata were removed. Historical installation cells and local data dependencies are retained. Review code before running; it can download models, install packages and start expensive training. See `docs/experiment_catalogue.md` and the new pipeline for supported experiments.


In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import os
import numpy as np
import textwrap
import pyarrow

In [ ]:
train_path = "data/historical/ecthr_a_train_articles_described.parquet"
test_path = "data/historical/ecthr_a_test.parquet"
validation_path = "data/historical/ecthr_a_validation.parquet"

In [ ]:
# Read the Parquet files into pandas DataFrame objects
train = pd.read_parquet(train_path)
test = pd.read_parquet(test_path)
validation = pd.read_parquet(validation_path)


In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
validation.head()

In [ ]:
test['articles'] = test['labels'].apply(lambda x: tuple(x))
test['articles'].unique()

In [ ]:
validation['articles'] = validation['labels'].apply(lambda x: tuple(x))
validation['articles'].unique()

In [ ]:
# Define the mapping dictionary for article numbers to descriptions
article_mapping = {
0: "Article 1: Obligation to respect human rights",
1: "Article 2: Right to life",
2: "Article 3: Prohibition of torture",
3: "Article 4: Prohibition of slavery and forced labour",
4: "Article 5: Right to liberty and security",
5: "Article 6: Right to a fair trial",
6: "Article 7: No punishment without law",
7: "Article 8: Right to respect for private and family life",
8: "Article 9: Freedom of thought, conscience and religion",
9: "Article 10: Right to freedom of speech and expression"
}

In [ ]:
ex = 5
print(train['labels_slashable'][ex])
print("\n ---Articles---")
print(train['labels'][ex])

In [ ]:
textwrap.wrap(train['text'][ex],width=140)

In [ ]:
def map_articles(articles_tuple):
    return [article_mapping[article] for article in articles_tuple if article in article_mapping]

In [ ]:
# Apply the mapping function
test['articles'] = test['labels'].apply(map_articles)
test

In [ ]:
# Apply the mapping function
validation['articles'] = validation['labels'].apply(map_articles)
validation

In [ ]:
# Define a function to check if a list is empty or contains NaN
def is_empty_or_nan(lst):
    return len(lst) == 0

# Count null or empty values in 'labels' column
null_or_empty_count_labels = train['labels'].apply(is_empty_or_nan).sum()
print(f"Null or empty values in train 'labels' column: {null_or_empty_count_labels}")

In [ ]:
# Count null or empty values in 'labels' column
null_or_empty_count_labels = test['labels'].apply(is_empty_or_nan).sum()
print(f"Null or empty values in test 'labels' column: {null_or_empty_count_labels}")

In [ ]:
# Count null or empty values in 'labels' column
null_or_empty_count_labels = validation['labels'].apply(is_empty_or_nan).sum()
print(f"Null or empty values in validation 'labels' column: {null_or_empty_count_labels}")

In [ ]:
# Define a function to replace empty lists with ["No Articles Violated"]
def replace_empty_with_special(lst):
    return ["No Articles Violated"] if len(lst) == 0 else lst

# Apply the function to both columns
#train['labels'] = train['labels'].apply(replace_empty_with_special)
train['labels_slashable'] = train['labels_slashable'].apply(replace_empty_with_special)
test['articles'] = test['articles'].apply(replace_empty_with_special)
validation['articles'] = validation['articles'].apply(replace_empty_with_special)

In [ ]:
train.rename(columns={'labels_slashable':'articles'},inplace=True)

In [ ]:
validation

In [ ]:
train

In [ ]:
# Define the function to create the instruction format
def create_instruction_format(row):
    input_text = row['text']
    labels = row['labels']
    articles = row['articles']
    # Convert each label to the "Article X" format and return as list of strings
    labels_list = [f"Article {label}" for label in labels]
    return {
        'text': input_text,
        'labels': labels_list,
        'articles': articles
    }

In [ ]:
# Apply the function to annotate each row
annotated_data = validation.apply(create_instruction_format, axis=1)
validation = pd.DataFrame(annotated_data.tolist())
validation

In [ ]:
# Apply the function to annotate each row
annotated_data = test.apply(create_instruction_format, axis=1)
test = pd.DataFrame(annotated_data.tolist())
test


#### List to np.array

In [ ]:
# Convert all elements in the 'articles' column to array objects

# Function to convert list of strings to numpy array with dtype=object
def convert_to_object_array(lst):
    return np.array(lst, dtype=object)

# Apply the function to the 'articles' column
train['articles'] = train['articles'].apply(convert_to_object_array)
test['articles'] = test['articles'].apply(convert_to_object_array)
validation['articles'] = validation['articles'].apply(convert_to_object_array)

In [ ]:
# Construct the full file path
file_path = os.path.join("data/historical/Article described ecthr_a","ecthr_a_article_desc_validation.parquet")
        
# Save to Parquet file
validation.to_parquet(file_path)

In [ ]:
# Construct the full file path
file_path = os.path.join("data/historical/Article described ecthr_a","ecthr_a_article_desc_test.parquet")
        
# Save to Parquet file
test.to_parquet(file_path)

In [ ]:
# Construct the full file path
file_path = os.path.join("data/historical/Article described ecthr_a","ecthr_a_article_desc_train.parquet")
        
# Save to Parquet file
train.to_parquet(file_path)

In [ ]:
ls = ["5. The applicant was born in 1940 and lives in Odesa.", 
      "6. At the time of the events the applicant was the director general of a joint venture V. (“company V.”), which had its office in the premises belonging to a joint stock company Y. (“company Y.”).",
      "7. In March 2001 the owner of company Y. changed. The new management questioned the legality of the use of its premises by company V. More specifically, they challenged the lease contract of 12 January 1999 in respect of those premises, which had been signed by the applicant, on the one side, and N., the chairman of the board of directors of company Y. at the time, on the other side. Under that contract, company V. could use the office space in question from 12 January 1999 to 12 January 2020 without any payment, but in exchange for certain services for company Y.",
      "8. Starting from April 2001, company Y. no longer allowed access to its premises to company V. As a result, the applicant transformed his flat in a temporary office of company V.",
      "9. In June 2001 company V. brought commercial proceedings against company Y. seeking compliance with the lease contract. Company Y., in turn, lodged a counter-claim seeking invalidation of that contract. By a final decision of the Supreme Court of 25 September 2003, the national courts rejected the claim of company V. and discontinued the proceedings as regards company Y.’s counter-claim. It was concluded that “there [was] no subject matter of the dispute”, given that the impugned contract failed to stipulate basic terms inherent in a lease contract and could not therefore be regarded as a lease contract.",
      "10. On 3 October 2001 a criminal case was opened in respect of suspected forgery of the lease contract of 12 January 1999, without being targeted against any particular persons.",
      "11. On 8 October 2001 the Odesa Prymorskyy District Prosecutor’s Office (“the Prymorskyy Prosecutor’s Office”) issued a warrant for seizure of fifteen documents relevant for the investigation, such as the original of the lease contract itself, related correspondence and several statements of acceptance of the services indicated in the contract (see paragraph 7 above). The seizure was to be carried out in company V.’s office.",
      "12. On 11 October 2001 the seizure took place in the applicant’s flat, in the presence of his wife. It appears that the applicant was not present. Eleven of the fifteen documents listed in the warrant were seized. The seizure report did not contain any information as to whether it had been handed to any person occupying the premises. The applicant did not specify in the domestic proceedings, or in the present proceedings, how the seizure of the documents had taken place.",
      "13. On 15 October 2001 the seizure warrant of 8 October 2001 was served on the applicant.",
      "14. On 22 October 2001 the investigator decided that a forensic expert examination of the signatures on the contract of 12 January 1999 was required in order to establish their real date.",
      "15. On 7 November 2001 the Odesa Prymorskyy District Court (“the Prymorskyy Court”) ordered a search of the applicant’s flat, which was also company V.’s office, with a view to collecting samples of his handwriting and signatures. As stated in the court’s ruling, “notebooks, correspondence and other personal records with [the applicant’s] handwriting” were required for the above-mentioned expert evaluation. That decision was not amenable to appeal.",
      "16. On the following day the search took place in the applicant’s flat in his presence and resulted in a seizure of eleven documents. The applicant did not provide any description, be it in the domestic proceedings or in the present proceedings, as to how the search had been carried out.",
      "17. On 24 December 2001 the investigator ordered a seizure of company V.’s constituent documents from the company’s office. It appears that the seizure was carried out on the same day in the applicant’s flat.",
      "18. On 9 January 2002 the above seizure warrant was served on the applicant.",
      "19. On 6 May 2002 company V. founders’ meeting decided to suspend the applicant from the exercise of his duties as its director general pending the ongoing criminal proceedings.",
      "20. On 20 August 2002 the prosecutor discontinued the proceedings for the absence of unequivocal evidence of a criminal offence. Although a forensic expert examination had established that the signatures on the impugned contract had been antedated (namely, it was established that they had been made no earlier than in February 2001), the official approval of the technical methods used by the expert was previewed only for the autumn of 2002.",
      "21. On 1 September 2002 the applicant resumed his duties in company V.",
      "22. On 28 August 2002 the applicant brought proceedings against the Prymorskyy Prosecutor’s Office claiming compensation in respect of non-pecuniary damage allegedly caused by its unlawful actions. The applicant based his lawsuit on the fact that the criminal proceedings had been terminated, without raising any specific complaints about the search and seizures. He contended that the institution of the criminal proceedings had been arbitrary, which had led, inter alia, to the unlawful search of his flat and the seizure of documents.",
      "23. On 9 December 2002 the Prymorskyy Court rejected the applicant’s claim as unfounded. The case file does not contain a copy of that decision. It appears that the court’s conclusion was that the applicant had not suffered any non-pecuniary damage.",
      "24. The applicant appealed. He argued, in particular, that the impugned measures had been devoid of any legitimate purpose given the impossibility at the time to carry out the forensic handwriting examination ordered by the investigator. He further submitted that, in ordering the seizure of documents, no differentiation had been made between the company’s premises and his home. The applicant maintained that the first-instance court had left those matters without consideration.",
      "25. On 11 September 2003 the Odesa Regional Court of Appeal rejected the applicant’s appeal. It held, in particular, that the company’s office had de facto been located in the applicant’s flat. As regards his complaint about the court’s failure to assess all the circumstances of the case, the appellate court dismissed it as ungrounded.",
      "26. On 3 February 2006 the Supreme Court upheld the lower courts’ decisions."]

In [ ]:
train

In [ ]:
test